In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
print(df.info())

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
cols = df.columns.drop("Target")

for col in cols:
  df[col] = df[col].fillna(df[col].mean())
df.isna().sum()

In [ ]:
# Task 2: Write your code here:
df.duplicated().sum()

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
categorical_cols

#There is no categorical columns !

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts()

# The target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target",axis=1).copy()
y = df["Target"].copy()

In [ ]:
%pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score

avgScore =[]
avgAcc =[]
sumAcc = 0
sumF1 = 0
model = CatBoostClassifier(n_estimators=100,verbose=0,max_depth=4, random_state=42)
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
y_pred =0
for fold_idx, (train_index, test_index) in enumerate(skf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    # Train
  model.fit(X_train,y_train)

  # Predict
  y_pred = model.predict(X_test)
  # Calculate evaluation metrics
  f1Score = f1_score(y_test, y_pred)
  accuracy = accuracy_score(y_test, y_pred)
  sumF1 = sumF1+f1Score
  sumAcc = sumAcc + accuracy
  print(f"F1 Score: {sumF1/n_splits} \nAccuracy: {sumAcc/n_splits}")
  # Store results
  avgAcc.append(accuracy)
  avgScore.append(f1Score)

print()
print("Avg Metrices:")
print(f"Avg F1 Score: {sumF1/n_splits} \nAvg Accuracy: {sumAcc/n_splits}")

In [ ]:
import matplotlib.pyplot as plt
feature_importance = pd.DataFrame({
    'feature': numerical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance=feature_importance[feature_importance["importance"] > 2.5]

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print("The golden feature is P_2")
print(df["P_2"].head())

In [ ]:
# Task Bonus: Write your code here: